In [2]:

import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE, dataset_tp_rp_split

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("data/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = [10]
selected_campaigns = list(range(1, 10))
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        "sinr",
        "rsrq"
    ],
    campaigns=selected_campaigns,
)




Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [3]:
from scripts.utils import extract_unique_npcis
from scripts.beamforming import *


In [4]:
from scripts.utils import haversine_distance
import pandas as pd


def wknn_one_tp_row(
        df_tp: pd.DataFrame,
        df_rp: pd.DataFrame,
        idx_sort: np.ndarray[int],
        W: np.ndarray[np.float64],
        k: int,
) -> (np.array, float):
    """
    :param df_tp: Dataframe of reference points - ASSUMED TO HAVE ONE ROW
    :param df_rp: Dataframe of test points
    :param idx_sort: sorted index matrix by weights
    :param W: the weight matrix for the test/reference points
    :param k: Number of neighbors for wKNN
    :return: Estimated locations and average error for the given k value
    """
    # Extract real positions of the test point (assuming only one row in df_tp)
    real_lat = df_tp["lat"].iloc[0]
    real_long = df_tp["lng"].iloc[0]
    real_position = np.array([[real_lat, real_long]])  # Still needs to be 2D for haversine_distance

    # Select the k-nearest reference points.
    # Since df_tp has one row, we only need the first row of idx_sort and W
    RFP_selected_idx = idx_sort[0, :k]
    W_row = W[0, :k]

    # Extract coordinates of the selected reference points
    lat_k_RFP_matrix = df_rp.iloc[RFP_selected_idx]["lat"].values
    long_k_RFP_matrix = df_rp.iloc[RFP_selected_idx]["lng"].values

    # Compute weighted sums of coordinates
    sum_lat = np.sum(lat_k_RFP_matrix * W_row)
    sum_long = np.sum(long_k_RFP_matrix * W_row)

    # Compute estimated coordinates of test points
    sum_weights = np.sum(W_row)
    lat_k_TP = np.where(sum_weights != 0, sum_lat / sum_weights, np.nan)
    long_k_TP = np.where(sum_weights != 0, sum_long / sum_weights, np.nan)

    # Compute errors using Haversine formula
    errors = haversine_distance(
        real_position[0, 0], real_position[0, 1], lat_k_TP, long_k_TP
    )

    # Store estimated locations
    TP_est_location = np.zeros((1, 2))  # Still 2D array for consistency in return type
    TP_est_location[0, 0] = lat_k_TP
    TP_est_location[0, 1] = long_k_TP

    return (
        TP_est_location,
        errors,
    )


In [29]:
import pandas as pd
import numpy as np
from scripts.matrix_operations import create_point_matrix, compute_weights


def beam_matching_strategy(df: pd.DataFrame, rf_param: RF_PARAM_5G, random: int) -> Tuple[
    float, float, Tuple[int, int], Tuple[int, int]]:
    # get the best beam for each point
    df['best_beam'] = df['measurements_matrix'].apply(
        lambda x: get_best_beam(x, rf_param)
    )

    df_tp, df_rp = dataset_tp_rp_split(df, 0.3, random)
    unique_npcis = extract_unique_npcis(df['measurements_matrix'])

    # Pre-compute reference point matrices by beam
    rp_matrices_by_beam = {}
    unique_beams = df_rp['best_beam'].unique()

    # Pre-compute matrices for each beam group
    for beam in unique_beams:
        beam_rps = df_rp[df_rp['best_beam'] == beam]
        m_rp, idx_rp = create_point_matrix(beam_rps, unique_npcis, rf_param)
        rp_matrices_by_beam[beam] = (m_rp, idx_rp, beam_rps)

    # Pre-compute the control matrix (all RPs) once
    m_rp_control, idx_rp_control = create_point_matrix(df_rp, unique_npcis, rf_param)

    data = []

    for i, (_, tp_row) in enumerate(df_tp.iterrows(), 1):
        tp = pd.DataFrame([tp_row])
        best_beam = tp_row['best_beam']

        # Get the pre-computed matrices for this beam
        if best_beam in rp_matrices_by_beam:
            m_rp, idx_rp, rps = rp_matrices_by_beam[best_beam]
        else:
            # Handle the case where the beam isn't in reference points
            rps = pd.DataFrame()  # Empty DataFrame
            m_rp, idx_rp = np.array([]), np.array([])

        # Create the point matrix for the test point
        m_tp, idx_tp = create_point_matrix(tp, unique_npcis, rf_param)

        # Compute weights only if we have matching RPs
        if len(rps) > 0:
            W, idx_sort = compute_weights(m_rp, idx_rp, m_tp, idx_tp)
            _, errors = wknn_one_tp_row(tp, rps, idx_sort, W, 2)
        else:
            continue

        # Compute control weights and errors
        W_control, idx_sort_control = compute_weights(m_rp_control, idx_rp_control, m_tp, idx_tp)
        _, errors_control = wknn_one_tp_row(tp, df_rp, idx_sort_control, W_control, 2)

        complexity = m_rp.shape[0] * m_rp.shape[1] if len(m_rp.shape) == 2 else None
        complexity_control = m_rp_control.shape[0] * m_rp_control.shape[1] if len(m_rp_control.shape) == 2 else None
        data.append([errors, errors_control, complexity, complexity_control])

    data = np.array(data)
    return data.mean(axis=0)


data = []

runs = 3

for i in range(runs):
    res = beam_matching_strategy(df, RF_PARAM_5G.RSRQ, 42 * i)
    data.append(res)
    print(f'\r{i}/{runs}')

res_df = pd.DataFrame(data, columns=['errors', 'errors_control', 'complexity', 'complexity_control'])

res_df

0/3
1/3
2/3


,errors,errors_control,complexity,complexity_control
0,6.861243,3.930089,47260.632911,634800.0
1,8.365591,4.141278,46197.632450,630200.0
2,7.856217,3.832029,46713.966387,633880.0


In [30]:
res_df.mean()

errors                     7.694351
errors_control             3.967799
complexity             46724.077249
complexity_control    632960.000000
dtype: float64

In [22]:
a = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

arr = [a for i in range(10)]

arr = np.array(arr).T

arr.mean(axis=0)

array([5.5, 5.5, 5.5, 5.5, 5.5, 5.5, 5.5, 5.5, 5.5, 5.5])